In [6]:
from rdkit import Chem
import torch
from protlig.helper import load_dataset

In [7]:
esol_data = load_dataset()

/Users/tungvuduc/projects/protlig/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ELEMENTS = ["C", "N", "O", "F", "S", "Cl", "Br", "I", "P"]
N_FEATURES = len(ELEMENTS) + 1 + 2 # one-hot + other + degree + aromatic
DTYPE = torch.float64

def mol_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    rows = []
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        onehot = [float(sym == e) for e in ELEMENTS]
        onehot.append(float(sym not in ELEMENTS))
        rows.append(onehot + [atom.GetDegree(), atom.GetIsAromatic()])
    x = torch.tensor(rows, dtype=DTYPE)

    src, dst = [], []

    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        src += [i, j]
        dst += [j, i]
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return x, edge_index, torch.tensor([y], dtype=DTYPE)

smiles = esol_data['SMILES'].iloc[0]
y = esol_data['label'].iloc[0]
x, edge_index, y = mol_to_graph(smiles, y)

In [ ]:
# def collate(graphs):

xs, ei, ys, batch = [], [], [], []
